In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import torch

In [2]:
from peft import LoraConfig, get_peft_model

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
model_name = "mistralai/Mistral-7B-v0.1"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:

data_path = '/content/drive/MyDrive/llm_finetuning/data/data.jsonl'
data = load_dataset("json",data_files=data_path)

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
def format_example(example):
    text = f"User:{example['input']}\nAssistant:{example['output']}"
    encoding = tokenizer(text, truncation=True, padding="max_length")
    encoding["labels"] = encoding["input_ids"].copy()
    return encoding

In [ ]:
tokanized_dataset = data.map(format_example)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/llm_finetuning/checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    logging_steps=5,
    save_steps=20,
    fp16=False,   # CPU → must be False
    report_to="none"
)

In [ ]:
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1
)

model = get_peft_model(model, config)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokanized_dataset["train"]
)

In [ ]:
trainer.train()

Step,Training Loss
5,4.724899
10,4.278717
15,3.986988


TrainOutput(global_step=15, training_loss=4.330201212565104, metrics={'train_runtime': 41.4481, 'train_samples_per_second': 1.448, 'train_steps_per_second': 0.362, 'total_flos': 37519594463232.0, 'train_loss': 4.330201212565104, 'epoch': 3.0})

In [ ]:
save_path = '/content/drive/MyDrive/llm_finetuning/model'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/llm_finetuning/model/tokenizer_config.json',
 '/content/drive/MyDrive/llm_finetuning/model/tokenizer.json')

In [ ]:
### Prediction on data

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = '/content/drive/MyDrive/llm_finetuning/model'

saved_tokenizer = AutoTokenizer.from_pretrained(model_path)
# saved_model = AutoModelForCausalLM.from_pretrained(model_path)

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(model_name)
saved_model = PeftModel.from_pretrained(base_model, model_path)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
def predict_intent(user_input):
    prompt = f"User:{user_input}\nAssistant:"

    inputs = saved_tokenizer(prompt, return_tensors="pt")

    outputs = saved_model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,   # deterministic output
        temperature=0.0
    )

    result = saved_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the answer part
    return result.split("Assistant:")[-1].strip()

In [ ]:
user_input = "I want to know my account balance"

prompt = f"User:{user_input}\nAssistant:"
inputs = saved_tokenizer(prompt, return_tensors="pt")

In [ ]:
inputs

{'input_ids': tensor([[    1,  1247, 28747, 28737,   947,   298,   873,   586,  2708,  7873,
            13,  7226, 11143, 28747]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
outputs = saved_model.generate(
  **inputs,
  max_new_tokens=10,
  do_sample=False,   # deterministic output
  temperature=0.0,
  pad_token_id=saved_tokenizer.eos_token_id # Fix the warning
)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
result = saved_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
result

'User:I want to know my account balance\nAssistant:I can help you with that.\nUser:'